In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

model.to(device)
model.eval()
def fariz_gpt(prompt, max_length=100, temperature=0.3, top_k=20):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)
question = "Explain transformer architecture in simple terms"
response = fariz_gpt(question)

print("Fariz-GPT:", response)


Fariz-GPT: Explain transformer architecture in simple terms.Only answer using factual information.

The following is a list of the most common problems with the transformer architecture.

The following is a list of the most common problems with the transformer architecture.

The following is a list of the most common problems with the transformer architecture.

The following is a list of the most common problems with the transformer architecture.

The following is a list of the most common problems with the transformer architecture.



In [ ]:
# -----------------------------
# 1️⃣ Install required libraries
# -----------------------------
# !pip install torch transformers pdfplumber sentencepiece --quiet

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from torch.optim import AdamW

import pdfplumber
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 2️⃣ Load and preprocess PDF
# -----------------------------
pdf_path = "transformers.pdf"  # your PDF path

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
    text = re.sub(r'\s+', ' ', text)
    return text

raw_text = extract_text_from_pdf(pdf_path)

# Split text into small chunks for training
chunk_size = 200  # number of characters per chunk
chunks = [raw_text[i:i+chunk_size] for i in range(0, len(raw_text), chunk_size)]

# -----------------------------
# 3️⃣ Tokenizer & Dataset
# -----------------------------
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT2 has no pad token

class PDFDataset(Dataset):
    def __init__(self, chunks, tokenizer, max_length=200):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []
        for chunk in chunks:
            encodings = self.tokenizer(chunk, truncation=True, max_length=max_length, padding='max_length')
            self.examples.append(torch.tensor(encodings['input_ids']))
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        x = self.examples[idx]
        return x, x  # input = output for language modeling

dataset = PDFDataset(chunks, tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# -----------------------------
# 4️⃣ Load GPT2 Model for Fine-Tuning
# -----------------------------
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))  # adjust token embeddings
model.to(device)
model.train()

# -----------------------------
# 5️⃣ Optimizer & Loss
# -----------------------------
optimizer = AdamW(model.parameters(), lr=5e-5)

# -----------------------------
# 6️⃣ Fine-Tune GPT
# -----------------------------
epochs = 3  # for demo, increase for better results
for epoch in range(epochs):
    total_loss = 0
    for input_ids, labels in dataloader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        outputs = model(input_ids, labels=labels)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

# -----------------------------
# 7️⃣ PDF-Grounded Fariz-GPT Inference
# -----------------------------
def fariz_gpt_pdf(prompt, max_length=150, temperature=0.3, top_k=20):
    model.eval()
    with torch.no_grad():
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        output = model.generate(
            inputs["input_ids"],
            max_length=max_length,
            temperature=temperature,
            top_k=top_k,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        return tokenizer.decode(output[0], skip_special_tokens=True)

# -----------------------------
# 8️⃣ Ask a question
# -----------------------------
question = "Explain transformer architecture in simple terms."
answer = fariz_gpt_pdf(question)
print("Fariz-GPT (PDF):", answer)


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


In [ ]:
!pip install gradio

In [ ]:


import gradio as gr

def chat_interface(question):
    return fariz_gpt_pdf(question)

iface = gr.Interface(fn=chat_interface, inputs="text", outputs="text", title="Fariz-GPT PDF Chatbot")
iface.launch()
